# Bài 33: Tối ưu hóa

Bài 31-32 đã dùng hạ gradient đơn thuần: lấy gradient, bước một bước kích thước cố định ngược chiều nó:

$$p \leftarrow p - \eta\,\nabla f(p)$$

trong đó $p$ là bất cứ thứ gì đang được học (một trọng số, một độ lệch, hoặc toàn bộ vector tham số cùng lúc), $\eta$ là **tốc độ học**, và $\nabla f(p)$ là gradient của hàm mất mát tại các tham số hiện tại. Như vậy là đủ để chứng minh *ý tưởng* hoạt động, nhưng hiếm khi đó là cách mạng thực tế được huấn luyện. Bài này trình bày bốn nâng cấp thực tế &mdash; **quán tính (momentum)**, **kích thước bước thích nghi (Adam)**, **các lược đồ khởi tạo trọng số**, và **lịch trình tốc độ học (learning rate schedules)** &mdash; mỗi cái khắc phục một kiểu thất bại cụ thể, rõ ràng của hạ gradient đơn thuần.

Lưu ý rằng trong huấn luyện thực tế, gradient hầu như không bao giờ được tính trên *toàn bộ* tập huấn luyện cùng lúc — nó sẽ không vừa bộ nhớ. Thay vào đó, ở mỗi bước nó được ước lượng từ một **lô nhỏ (mini-batch)** được lấy mẫu ngẫu nhiên (chẳng hạn 32 hoặc 256 mẫu). Ước lượng nhiễu, thay đổi theo từng lần lấy mẫu thu được chính là điều mang lại tên gọi cho **hạ gradient ngẫu nhiên (SGD)**. Bài này chủ yếu dùng gradient chính xác, không nhiễu của một hàm cố định (gọi nó là "SGD" cho tiện); mục cuối mô phỏng trực tiếp nhiễu lô nhỏ để cho thấy tại sao nó quan trọng.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

## Một địa hình hàm mất mát mà hạ gradient đơn thuần gặp khó

$f(x,y) = 0.05x^2 + 5y^2$ là một cái bát dốc theo $y$ và thoải theo $x$. Một kích thước bước đủ lớn để tiến triển dọc theo $x$ sẽ vượt quá đích dọc theo $y$, gây ra hiện tượng *zigzag* kinh điển.

In [ ]:
def f(p):
    return 0.05 * p[0]**2 + 5 * p[1]**2

def grad_f(p):
    return np.array([0.1 * p[0], 10 * p[1]])

start = np.array([-4.0, 1.0])
lr = 0.15

p = start.copy()
sgd_path = [p.copy()]
for _ in range(30):
    p = p - lr * grad_f(p)
    sgd_path.append(p.copy())
sgd_path = np.array(sgd_path)

print(f'plain SGD, 30 steps: final loss = {f(sgd_path[-1]):.4f}')

In [ ]:
xs = np.linspace(-4.5, 1, 200)
ys = np.linspace(-1.5, 1.5, 200)
XX, YY = np.meshgrid(xs, ys)
ZZ = 0.05 * XX**2 + 5 * YY**2

plt.contour(XX, YY, ZZ, levels=20, colors='lightgray', linewidths=0.7)
plt.plot(*sgd_path.T, '-o', markersize=3, color='tab:blue')
plt.scatter([0], [0], marker='*', s=150, color='black', zorder=5, label='minimum')
plt.legend(fontsize=8)
plt.title('Plain gradient descent: the classic zigzag')
plt.show()

## Quán tính: nhớ xem bạn đang hướng về đâu

Thay vì bước thuần túy theo gradient hiện tại, hãy tích lũy một **vận tốc (velocity)** tức thời &mdash; trung bình có trọng số của các gradient quá khứ &mdash; và bước theo đó:

$$v \leftarrow \beta v - \eta\,\nabla f(p), \qquad p \leftarrow p + v$$

Lợi ích thực sự của quán tính là tăng tốc, không phải giảm chấn. Dọc theo một hướng nhất quán (như trục $x$ thoải ở đây), mọi gradient đều chỉ cùng một hướng, nên vận tốc tích lũy tiếp tục tăng qua từng bước &mdash; kích thước bước hiệu dụng dọc theo trục đó trở nên lớn hơn nhiều so với chỉ riêng $\eta$, y như một quả bóng tăng tốc khi lăn xuống dốc. Dọc theo một hướng dao động (trục $y$ dốc, đổi dấu ở mỗi bước), các gradient liên tiếp triệt tiêu một phần trong cùng phép trung bình tích lũy đó &mdash; nhưng chỉ *một phần*: một hệ số quán tính duy nhất $\beta$ được dùng chung cho mọi hướng, nên sự triệt tiêu đó không được điều chỉnh theo độ cong của riêng hướng nào, và trục dốc vẫn có thể dao động đáng kể &mdash; đôi khi còn nhiều hơn cả dao động của hạ gradient đơn thuần &mdash; trước khi tắt dần. Hiệu quả chung, qua cả một lần chạy, thường vẫn là có lợi: tiến triển nhanh hơn trên trục hành xử tốt bù đắp cho trục được giảm chấn không hoàn hảo. Nhưng đó là thắng lợi nhờ tăng tốc cộng với một cái phanh thô, chưa được điều chỉnh, chứ không phải một cách khắc phục gọn gàng cho dao động &mdash; đó mới là thứ Adam, tiếp theo, thực sự được xây dựng để làm.

(Quán tính có lợi ích thứ hai còn quan trọng hơn trong huấn luyện thực tế: với gradient lô nhỏ có nhiễu thay vì gradient chính xác dùng ở đây, phép trung bình tích lũy hoạt động như một bộ lọc thông thấp, làm trơn nhiễu giữa các lần lấy mẫu thành một hướng đáng tin cậy hơn. Ta sẽ thấy trực tiếp trường hợp có nhiễu gần cuối bài.)

In [ ]:
momentum = 0.9
p = start.copy()
v = np.zeros(2)
mom_path = [p.copy()]
for _ in range(30):
    g = grad_f(p)
    v = momentum * v - lr * g
    p = p + v
    mom_path.append(p.copy())
mom_path = np.array(mom_path)

print(f'SGD + momentum, 30 steps: final loss = {f(mom_path[-1]):.4f}')

### Đối chiếu với PyTorch

In [ ]:
p_t = torch.tensor(start.copy(), requires_grad=True)
opt = torch.optim.SGD([p_t], lr=lr, momentum=momentum)
for _ in range(30):
    opt.zero_grad()
    loss = 0.05 * p_t[0]**2 + 5 * p_t[1]**2
    loss.backward()
    opt.step()

print(f'our result:   {mom_path[-1]}')
print(f'torch result: {p_t.detach().numpy()}')
print(f'max diff: {np.abs(p_t.detach().numpy() - mom_path[-1]).max():.2e}')

## Adam: kích thước bước thích nghi theo từng tham số

**Adam** (<a href="../references.html#kingma-ba-2014">Kingma & Ba, 2014</a><span class="landmark-paper">&#9733;</span>) theo dõi cả một trung bình tích lũy kiểu quán tính của gradient ($m$) *lẫn* một trung bình tích lũy của bình phương gradient ($v$), rồi chia bước cho $\sqrt{v}$ &mdash; tự động thu nhỏ kích thước bước cho các tham số có gradient lớn nhất quán (như hướng $y$ dốc ở đây) và tăng nó cho các tham số có gradient nhỏ:

$$m \leftarrow \beta_1 m + (1-\beta_1)g, \qquad v \leftarrow \beta_2 v + (1-\beta_2)g^2, \qquad p \leftarrow p - \eta\frac{\hat{m}}{\sqrt{\hat{v}}+\epsilon}$$

($\hat{m}, \hat{v}$ là các phiên bản đã hiệu chỉnh độ chệch (bias-corrected) của $m, v$, chủ yếu quan trọng trong vài bước đầu tiên.)

In [ ]:
beta1, beta2, adam_eps = 0.9, 0.999, 1e-8
adam_lr = 0.3

p = start.copy()
m, v_sq = np.zeros(2), np.zeros(2)
adam_path = [p.copy()]
for t in range(1, 31):
    g = grad_f(p)
    m = beta1 * m + (1 - beta1) * g
    v_sq = beta2 * v_sq + (1 - beta2) * g**2
    m_hat = m / (1 - beta1**t)
    v_hat = v_sq / (1 - beta2**t)
    p = p - adam_lr * m_hat / (np.sqrt(v_hat) + adam_eps)
    adam_path.append(p.copy())
adam_path = np.array(adam_path)

p_t2 = torch.tensor(start.copy(), requires_grad=True)
opt2 = torch.optim.Adam([p_t2], lr=adam_lr, betas=(beta1, beta2), eps=adam_eps)
for _ in range(30):
    opt2.zero_grad()
    loss = 0.05 * p_t2[0]**2 + 5 * p_t2[1]**2
    loss.backward()
    opt2.step()

print(f'Adam, 30 steps: final loss = {f(adam_path[-1]):.4f}')
print(f'max diff vs torch.optim.Adam: {np.abs(p_t2.detach().numpy() - adam_path[-1]).max():.2e}')

### Cả ba đường đi, trực quan hóa

In [ ]:
plt.contour(XX, YY, ZZ, levels=20, colors='lightgray', linewidths=0.7)
plt.plot(*sgd_path.T, '-o', markersize=3, label=f'plain SGD (loss={f(sgd_path[-1]):.3f})')
plt.plot(*mom_path.T, '-o', markersize=3, label=f'momentum (loss={f(mom_path[-1]):.3f})')
plt.plot(*adam_path.T, '-o', markersize=3, label=f'Adam (loss={f(adam_path[-1]):.3f})')
plt.scatter([0], [0], marker='*', s=150, color='black', zorder=5, label='minimum')
plt.legend(fontsize=8)
plt.title('30 steps of each optimizer on the same narrow bowl')
plt.show()

SGD đơn thuần bị kẹt dao động qua thung lũng hẹp, hầu như không tiến triển dọc theo hướng thoải. Vận tốc tích lũy của quán tính tăng tốc rõ rệt tiến triển dọc theo hướng thoải đó, và kết thúc với hàm mất mát thấp hơn SGD đơn thuần &mdash; nhưng hãy để ý rằng đường đi của nó thực ra dao động *rộng hơn* theo $y$ so với chính SGD đơn thuần, chứ không hẹp hơn, vì một hệ số quán tính dùng chung duy nhất không thể giảm chấn mỗi trục đến mức mà riêng trục đó cần. Adam, vốn co giãn lại từng hướng một cách độc lập, mới là thứ thực sự chế ngự trực tiếp trục dốc, trong khi vẫn lao nhanh dọc theo trục thoải &mdash; đạt hàm mất mát thấp nhất trong ba cách chỉ trong cùng 30 bước.

Ảnh động này cho ba bộ tối ưu cùng đua từng bước, từ cùng điểm xuất phát, trên cùng một cái bát. Hãy quan sát đường zigzag nhỏ, chậm của SGD đơn thuần đình trệ còn cách xa cực tiểu; quán tính tăng tốc dọc theo trục thoải nhưng dao động ngày càng dữ dội dọc theo trục dốc, vượt qua cực tiểu hơn một lần; và Adam bình tĩnh giảm chấn trục dốc trong khi vẫn tiến nhanh dọc theo trục thoải, kết thúc gần cực tiểu nhất.

<img src="../animations/optimizer_race_demo.gif" alt="Ảnh động cuộc đua giữa SGD đơn thuần, quán tính và Adam đi xuống cùng một địa hình hàm mất mát hình bát hẹp từng bước từ cùng điểm xuất phát, với SGD đơn thuần đình trệ, quán tính vượt quá đích, và Adam hội tụ gần cực tiểu nhất" style="display: block; margin: 0 auto; max-width: 680px;">

## Khởi tạo trọng số: tại sao số 0 không hoạt động

Thật hấp dẫn khi khởi tạo mọi trọng số bằng 0. Dù đây có vẻ là điểm xuất phát "trung lập" nhất, nó thực ra là thảm họa với bất kỳ lớp nào có nhiều hơn một đơn vị: mọi đơn vị ẩn tính chính xác cùng một hàm của đầu vào, nhận chính xác cùng một gradient, và được cập nhật chính xác cùng một lượng, mãi mãi. **Tính đối xứng (symmetry)** này không bao giờ tự phá vỡ.

In [ ]:
rng = np.random.default_rng(0)
theta_in = rng.uniform(0, 2 * np.pi, 60)
inner = np.stack([0.5 * np.cos(theta_in), 0.5 * np.sin(theta_in)], axis=1) + rng.normal(0, 0.1, (60, 2))
theta_out = rng.uniform(0, 2 * np.pi, 60)
outer = np.stack([2.0 * np.cos(theta_out), 2.0 * np.sin(theta_out)], axis=1) + rng.normal(0, 0.15, (60, 2))
X = np.vstack([inner, outer])
y = np.concatenate([np.zeros(60), np.ones(60)])

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def relu(z):
    return np.maximum(0, z)

def relu_deriv(z):
    return (z > 0).astype(np.float64)

H = 4

def train_mlp(W1, b1, W2, b2, n_epochs=3000, lr=0.1):
    for _ in range(n_epochs):
        z1 = X @ W1 + b1
        a1 = relu(z1)
        z2 = (a1 @ W2 + b2).ravel()
        p = sigmoid(z2)
        n = len(y)
        grad_z2 = ((p - y) / n).reshape(-1, 1)
        grad_W2 = a1.T @ grad_z2
        grad_b2 = grad_z2.sum(axis=0)
        grad_a1 = grad_z2 @ W2.T
        grad_z1 = grad_a1 * relu_deriv(z1)
        grad_W1 = X.T @ grad_z1
        grad_b1 = grad_z1.sum(axis=0)
        W2 -= lr * grad_W2; b2 -= lr * grad_b2
        W1 -= lr * grad_W1; b1 -= lr * grad_b1
    return W1, b1, W2, b2, p

W1_zero, b1_zero = np.zeros((2, H)), np.zeros(H)
W2_zero, b2_zero = np.zeros((H, 1)), np.zeros(1)
W1_zero, b1_zero, W2_zero, b2_zero, p_zero = train_mlp(W1_zero, b1_zero, W2_zero, b2_zero)
acc_zero = ((p_zero > 0.5).astype(float) == y).mean()

rng2 = np.random.default_rng(8)
W1_rand = rng2.normal(size=(2, H)) * 0.7
W2_rand = rng2.normal(size=(H, 1)) * 0.7
b1_rand, b2_rand = np.zeros(H), np.zeros(1)
_, _, _, _, p_rand = train_mlp(W1_rand, b1_rand, W2_rand, b2_rand)
acc_rand = ((p_rand > 0.5).astype(float) == y).mean()

print(f'zero-initialized:   accuracy = {acc_zero:.1%}')
print(f'randomly initialized: accuracy = {acc_rand:.1%}')
print()
print('all 4 hidden units still have identical weight vectors after 3000 steps of zero-init training:')
print(np.round(W1_zero, 4))

Với khởi tạo bằng 0, cả bốn đơn vị ẩn bị khóa ở đúng 0 mãi mãi &mdash; mô hình không bao giờ thoát khỏi nghiệm tầm thường (ngang mức đoán ngẫu nhiên), bất kể huấn luyện bao lâu. Khởi tạo *ngẫu nhiên* nhỏ phá vỡ tính đối xứng: mỗi đơn vị bắt đầu tính một thứ hơi khác nhau, nên hạ gradient có thể đẩy chúng ra xa nhau và để chúng chuyên biệt hóa. Đây là lý do khởi tạo lớp mặc định của mọi framework đều dùng các giá trị ngẫu nhiên nhỏ, không bao giờ là số 0.

## Các lược đồ khởi tạo trọng số: "ngẫu nhiên nhỏ" nên nhỏ đến đâu?

Khởi tạo bằng 0 thất bại hoàn toàn; "ngẫu nhiên nhỏ" khắc phục được. Nhưng nhỏ *đến đâu*? Chọn một độ lệch chuẩn cố định thì nó hoạt động ở một độ rộng lớp, rồi âm thầm thất bại ở độ rộng khác. Hãy theo dõi phương sai của các giá trị kích hoạt sau khi cho dữ liệu ngẫu nhiên đi qua một chồng sâu các lớp tuyến tính dùng hàm kích hoạt `tanh`, dùng cùng một độ lệch chuẩn trọng số cố định ở mọi độ rộng:

In [ ]:
def forward_variance(width, std, depth=20):
    x = torch.randn(100, width)
    for _ in range(depth):
        W = torch.randn(width, width) * std
        x = torch.tanh(x @ W)
    return x.var().item()

print('fixed weight std = 0.05, activation variance after 20 tanh layers:')
for width in [16, 64, 256]:
    print(f'  width={width:4d}: variance = {forward_variance(width, std=0.05):.2e}')

Độ lệch chuẩn cố định 0.05 biến mất về cơ bản bằng 0 sau 20 lớp — và mức độ biến mất phụ thuộc mạnh vào độ rộng lớp. Phương sai trước kích hoạt của mỗi lớp tỉ lệ với `fan_in * std^2`, trong đó **fan-in** là số đầu vào của lớp: ở độ rộng 16, đó là `16 * 0.05^2 ≈ 0.04`, nên mỗi lớp chỉ giữ lại khoảng 4% phương sai của lớp trước, và 20 lớp cộng dồn xóa sạch tín hiệu gần như hoàn toàn. Ở độ rộng 256, *cùng* độ lệch chuẩn cố định đó giữ lại `256 * 0.05^2 ≈ 0.64` phương sai mỗi lớp — vẫn thu nhỏ, nhưng chậm hơn nhiều, đó chính là lý do cột độ rộng 256 ở trên biến mất ít nghiêm trọng hơn nhiều so với cột độ rộng 16. Tác động của một độ lệch chuẩn cố định phụ thuộc hoàn toàn vào fan-in, nên cùng con số làm biến mất nghiêm trọng một lớp hẹp, ở một lớp đủ rộng, sẽ đẩy `fan_in * std^2` vượt quá 1 và bắt đầu *bùng nổ* thay vào đó.

**Khởi tạo Xavier** (<a href="../references.html#glorot-bengio-2010">Glorot & Bengio, 2010</a>) &mdash; còn gọi là *khởi tạo Glorot* &mdash; khắc phục điều này bằng cách co giãn độ lệch chuẩn theo fan-in của lớp: `std = sqrt(1 / fan_in)`, giữ `fan_in * std^2` cố định đúng bằng 1 bất kể độ rộng. Lặp lại cùng thí nghiệm với trọng số được co giãn theo Xavier:

In [ ]:
def forward_variance_xavier(width, depth=20):
    x = torch.randn(100, width)
    for _ in range(depth):
        std = (1.0 / width) ** 0.5
        W = torch.randn(width, width) * std
        x = torch.tanh(x @ W)
    return x.var().item()

print('Xavier std = sqrt(1/fan_in), activation variance after 20 tanh layers:')
for width in [16, 64, 256]:
    print(f'  width={width:4d}: variance = {forward_variance_xavier(width):.3f}')

# check the formula against PyTorch's own implementation
layer = nn.Linear(256, 256, bias=False)
nn.init.xavier_normal_(layer.weight)
expected_std = (2.0 / (256 + 256)) ** 0.5  # nn.init's xavier_normal_ uses fan_in AND fan_out
print(f'\nnn.init.xavier_normal_ weight std: {layer.weight.std().item():.4f}  (formula predicts {expected_std:.4f})')

Xavier giữ phương sai ở cùng một khoảng bất kể độ rộng, thay vì biến mất một cách thảm hại. Nhưng phép suy ra của Xavier giả định một hàm kích hoạt đối xứng như `tanh`.

**Khởi tạo Kaiming** (<a href="../references.html#he-2015-kaiming">He et al., 2015</a>) &mdash; còn gọi là *khởi tạo He* &mdash; là cùng ý tưởng đó được điều chỉnh cho `ReLU`, hàm đưa khoảng một nửa đầu vào về 0 và vì vậy cần gấp đôi phương sai để bù lại: `std = sqrt(2 / fan_in)`. Dùng tỉ lệ suy ra từ `tanh` của Xavier trên một mạng `ReLU` vẫn bị biến mất:

In [ ]:
def forward_variance_relu(width, depth=20, kaiming=True):
    x = torch.randn(100, width)
    for _ in range(depth):
        std = (2.0 / width) ** 0.5 if kaiming else (1.0 / width) ** 0.5
        W = torch.randn(width, width) * std
        x = torch.relu(x @ W)
    return x.var().item()

v_xavier_on_relu = forward_variance_relu(256, kaiming=False)
v_kaiming = forward_variance_relu(256, kaiming=True)
print(f'ReLU net, Xavier scale (sqrt(1/fan_in), no ReLU correction): variance after 20 layers = {v_xavier_on_relu:.2e}')
print(f'ReLU net, Kaiming scale (sqrt(2/fan_in)):                    variance after 20 layers = {v_kaiming:.3f}')

layer2 = nn.Linear(256, 256, bias=False)
nn.init.kaiming_normal_(layer2.weight, nonlinearity='relu')
expected_std2 = (2.0 / 256) ** 0.5
print(f'\nnn.init.kaiming_normal_ weight std: {layer2.weight.std().item():.4f}  (formula predicts {expected_std2:.4f})')

Quy tắc ngón tay cái: dùng khởi tạo Kaiming cho các mạng họ `ReLU` (mặc định cho `nn.Conv2d`/`nn.Linear` thực ra đã là một biến thể của nó), và Xavier cho `tanh`/`sigmoid`. Cả hai đều tốt hơn hẳn việc chọn một độ lệch chuẩn cố định rồi hy vọng. Chúng được suy ra, không phải tinh chỉnh, từ một yêu cầu đơn giản: giữ phương sai kích hoạt gần như không đổi từ lớp này sang lớp khác, để một mạng có thể được làm sâu tùy ý mà không âm thầm mất tín hiệu ngay cả trước khi huấn luyện bắt đầu.

## Lịch trình tốc độ học

Một tốc độ học cố định duy nhất cho cả quá trình huấn luyện thật hấp dẫn — đó là mặc định hiển nhiên — nhưng nó thường là lựa chọn sai: một tốc độ đủ lớn để tiến triển nhanh lúc đầu thường quá lớn để dừng lại chính xác khi huấn luyện tiến gần một cực tiểu. **Lịch trình tốc độ học** thay đổi tốc độ theo thời gian. Ba lịch trình phổ biến, được đối chiếu với `torch.optim.lr_scheduler`:

- **Giảm theo bậc (step decay)**: nhân tốc độ với một hệ số cố định mỗi `N` bước.
- **Ủ cosin (cosine annealing)**: giảm mượt mà theo một đường cosin từ tốc độ ban đầu xuống (gần) 0 — cái tên được mượn trực tiếp từ **ủ mô phỏng (simulated annealing)**, kỹ thuật tối ưu hóa cổ điển bắt đầu "nóng" (tự do khám phá, thậm chí chấp nhận các bước đi tệ hơn) rồi dần "nguội" về một nghiệm cố định.
- **Khởi động tuyến tính (linear warmup)**: *tăng dần* từ 0 qua vài bước đầu tiên, trước khi áp dụng lịch trình chính &mdash; được dùng vì các cập nhật ban đầu, trước khi thống kê gradient của mô hình ổn định, có thể không đáng tin đến mức một tốc độ lớn ngay từ bước đầu gây ra thiệt hại mà vài bước tăng dần đã có thể tránh được.

In [ ]:
base_lr = 0.5

def check_schedule(name, manual_fn, torch_scheduler_factory, steps):
    p = torch.tensor([1.0], requires_grad=True)
    opt = torch.optim.SGD([p], lr=base_lr)
    sched = torch_scheduler_factory(opt)
    torch_lrs = []
    for step in range(steps):
        torch_lrs.append(opt.param_groups[0]['lr'])
        opt.step()
        sched.step()
    manual_lrs = [manual_fn(step) for step in range(steps)]
    max_diff = max(abs(a - b) for a, b in zip(torch_lrs, manual_lrs))
    print(f'{name}: max diff vs torch.optim.lr_scheduler = {max_diff:.2e}')
    return manual_lrs

step_lrs = check_schedule(
    'step decay',
    lambda step: base_lr * (0.5 ** (step // 5)),
    lambda opt: torch.optim.lr_scheduler.StepLR(opt, step_size=5, gamma=0.5),
    steps=20)

cos_lrs = check_schedule(
    'cosine annealing',
    lambda step: 0.5 * base_lr * (1 + np.cos(np.pi * step / 20)),
    lambda opt: torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=20),
    steps=20)

warmup_steps = 5
warmup_lrs = check_schedule(
    'linear warmup',
    lambda step: base_lr * min(1.0, (step + 1) / warmup_steps),
    lambda opt: torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda=lambda step: min(1.0, (step + 1) / warmup_steps)),
    steps=10)

plt.figure(figsize=(6, 4))
plt.plot(step_lrs, '-o', markersize=3, label='step decay')
plt.plot(cos_lrs, '-o', markersize=3, label='cosine annealing')
plt.plot(warmup_lrs, '-o', markersize=3, label='linear warmup')
plt.xlabel('step'); plt.ylabel('learning rate'); plt.legend(fontsize=8)
plt.title('Three learning rate schedules')
plt.show()

### Tại sao giảm dần thực sự có ích: gradient có nhiễu

Trên cái bát hẹp chính xác, không nhiễu ở đầu bài, một tốc độ học hằng số được chọn tốt hội tụ hoàn toàn ổn — không có gì để phải giảm dần. Tuy nhiên, như đã nêu ở trên, huấn luyện thực tế hầu như không bao giờ thấy gradient chính xác: mỗi bước dùng một ước lượng lô nhỏ, vốn có nhiễu. Hãy mô phỏng điều đó bằng cách thêm nhiễu ngẫu nhiên vào gradient ở mỗi bước, và so sánh một tốc độ hằng số với giảm theo bậc.

In [ ]:
def run_noisy(schedule_fn, steps=400, seed=0, noise_std=0.6):
    rng = np.random.default_rng(seed)
    p = np.array([-4.0, 1.0])
    losses = []
    for t in range(steps):
        noisy_g = grad_f(p) + rng.normal(0, noise_std, size=2)  # stand-in for mini-batch noise
        p = p - schedule_fn(t) * noisy_g
        losses.append(f(p))
    return losses

noisy_lr = 0.18
const_losses = run_noisy(lambda t: noisy_lr)
decay_losses = run_noisy(lambda t: noisy_lr * (0.3 ** (t // 80)))

print(f'constant lr:  mean loss over final 50 steps = {np.mean(const_losses[-50:]):.4f}')
print(f'step decay:   mean loss over final 50 steps = {np.mean(decay_losses[-50:]):.4f}')

plt.figure(figsize=(6, 4))
plt.semilogy(const_losses, label='constant lr', alpha=0.8)
plt.semilogy(decay_losses, label='step decay', alpha=0.8)
plt.xlabel('step'); plt.ylabel('loss (log scale)'); plt.legend(fontsize=8)
plt.title('Noisy gradients: constant lr vs. step decay')
plt.show()

Với gradient có nhiễu, một tốc độ học hằng số đủ lớn để tiến triển nhanh lúc đầu không bao giờ thực sự ổn định — nó cứ nảy quanh cực tiểu một lượng tỉ lệ với chính tốc độ học, mãi mãi. Giảm theo bậc đạt được cùng tiến triển nhanh lúc đầu, rồi thu nhỏ tốc độ khi huấn luyện tiếp tục, thu hẹp biên độ nảy đó và dừng lại gần cực tiểu thực hơn nhiều. Đây là lý lẽ thực sự cho lịch trình tốc độ học: không phải vì một tốc độ cố định là "sai" trên một bài toán lý tưởng không nhiễu nào đó, mà vì nó không thể đồng thời lớn (để nhanh) và nhỏ (để chính xác) khi mọi gradient đều là một ước lượng có nhiễu, vốn là tình huống bình thường của huấn luyện theo lô nhỏ thực tế.

### Bài tập

1. Tăng `momentum` từ 0.9 lên 0.99 trên bài toán bát hẹp. Nó hội tụ nhanh hơn, hay bắt đầu vượt quá cực tiểu và dao động trên trục *thoải* lúc này?
2. Thử khởi tạo trọng số của MLP giống hệt nhau nhưng *khác không* (ví dụ `np.full((2, H), 0.3)` cho cả hai lớp) thay vì toàn 0. Tính đối xứng có vẫn không bị phá vỡ không? Tại sao bạn kỳ vọng cùng kiểu thất bại từ bất kỳ khởi tạo nào mà mọi đơn vị ẩn bắt đầu giống hệt nhau, không chỉ khởi tạo toàn 0?
3. `adam_lr` (0.3) của Adam lớn hơn nhiều so với `lr` (0.15) của SGD đơn thuần trong notebook này, vậy mà Adam vẫn ổn định trong khi một lần chạy SGD đơn thuần với `lr=0.3` sẽ phân kỳ dữ dội trên trục dốc. Hãy thử và xác nhận. Điều đó gợi ý gì về *lý do* Adam thường được mô tả là dễ dãi hơn với lựa chọn tốc độ học?
4. `forward_variance_relu` so sánh tỉ lệ Xavier với Kaiming ở `width=256`. Chạy lại ở `width=16` và `width=1024`. *Khoảng cách* giữa hai cách lớn hơn hay nhỏ hơn khi độ rộng tăng, và điều đó có khớp với lập luận co giãn theo fan-in dùng để suy ra cả hai công thức không?
5. Trong ví dụ tốc độ học với gradient có nhiễu, thử `noise_std=0.0` (hoàn toàn không có nhiễu). Giảm theo bậc có còn giúp ích, gây hại, hay không tạo ra khác biệt so với tốc độ hằng số — và điều đó có khớp với khẳng định rằng lịch trình quan trọng *vì* nhiễu gradient, chứ không phải bất chấp bản thân bề mặt hàm mất mát không?